# Savant v2 (ML_model_v2) — classifier + regressor, esmc-300m + chemberta-77M-MLM

The `savant1.py` two-stage architecture (a classifier gates binder/non-binder, a regressor
predicts pKd for everything, absolute pKd is the deliverable) rebuilt with:

- **esmc-300m** (`Synthyra/ESMplusplus_small`) and **ChemBERTa-77M-MLM** as the encoders -
  fixed, not swept. The sweep is done: under a leakage-controlled split every protein
  encoder from 8M to 1.5B parameters landed within 0.04 of every other, and no ligand
  encoder beat plain RDKit+ECFP4. Re-sweeping here would spend a training run re-deriving
  a conclusion already measured twice.
- **Physicochemical complementarity features** (section 4) - new. RDKit descriptors and
  ECFP4 describe the ligand; AA composition describes the protein; neither describes
  whether the two *fit* each other. This adds cheap, sequence/SMILES-only proxies for
  that: does the ligand's charge match the pocket's, do its H-bond groups have somewhere
  to go, does its size suit the protein it's binding.
- **The honest split** (section 6) - protein AND scaffold double-disjoint, with the
  straddling cells kept as the off-diagonal rather than discarded, exactly as
  `savant_v2/`'s step 7 and this project's later 2x2 established is necessary. A plain
  split overstates generalisation badly enough to invert model comparisons.
- **input.csv plus new Kd/Ki data**, if `ML_model_v2/data/expanded_kd_ki.csv` exists
  (built by `scripts/get_more_data.py`). Falls back to input.csv alone with a clear
  warning if it does not - this notebook does not block on that script having been run.

## What this deliberately does NOT do

- **No offset+residual decomposition.** That was tried and it lost to a single joint
  model in all four cells of the honest split, twice. The classifier/regressor split is
  orthogonal to that (binder gate vs everything-else, not target-level vs pair-level) and
  is kept because `savant1.py`'s own ablation showed it costs nothing and buys a free
  triage step.
- **No re-sweep of encoders.** Already measured; see above.
- **No IC50.** Kept out of both input.csv's own convention and `get_more_data.py`'s new
  data, on purpose - see that script's docstring for why.

## Two regressor objectives, same features, same split, reported side by side

The user's primary goal is absolute pKd; screening/ranking is explicitly the fallback if
that does not beat the field. Rather than pick one up front, this notebook trains **both**
on identical inputs:

    reg:squarederror              optimises absolute pKd directly
    rank:pairwise (qid=protein)   optimises within-target order directly

The pooled squared-error objective is corrupted by exactly the between-target noise this
project's own benchmark decomposition showed is close to unmeasurable (corpus-median vs
benchmark-target-mean r = -0.17); a ranking loss sidesteps it by construction. Reporting
both, on the same honest split, is a direct, nearly-free answer to which framing the data
actually supports - no guessing required.

## Output

A bundle in the same schema `ML_model/litpcba/prep_litpcba.py score` already reads
(`classifier_json`, `rdkit_columns`, `ecfp_bits`, `protein_encoder`, `ligand_encoder`), so
the real screening-deck evaluation built for the previous model runs against this one
with no changes.

In [1]:
import gc, json, re, time
from pathlib import Path

import numpy as np
import pandas as pd

HERE   = Path.cwd()                      # run from ML_model_v2/
REPO   = HERE.parent
SRC    = REPO / "input" / "input.csv"
EXTRA  = HERE / "data" / "expanded_kd_ki.csv"          # from scripts/get_more_data.py
BENCH  = REPO / "ML_model" / "benchmark" / "eval" / "output" / "bench_scored.csv"
OUT    = HERE / "work"
EMB    = OUT / "emb"
for d in (OUT, EMB):
    d.mkdir(parents=True, exist_ok=True)

SEED = 1337
MIN_PKD, MAX_PKD = 4.0, 12.0
MAX_LABEL_SD = 1.5
BINDER_PKD = 6.0
CENSOR_CAPS_NM = [10000.0, 100000.0, 1000000.0, 10000000.0]
USE_CAPS_AS_NEGATIVES = True

PROTEIN_ENC = dict(slug="esmc-300m", hf="Synthyra/ESMplusplus_small")
LIGAND_ENC  = dict(slug="chemberta-mlm77", hf="DeepChem/ChemBERTa-77M-MLM")

TEST_PROTEIN_FRAC, TEST_SCAFFOLD_FRAC = 0.20, 0.20
INDOMAIN_FRAC = 0.05
MIN_LIG_FOR_OFFSET = 10        # kept only for the diagnostic in section 6, not a feature
N_THREADS = -1
ECFP_BITS = 1024
AA = "ACDEFGHIKLMNPQRSTVWY"

def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
log(f"input.csv  {SRC}")
log(f"extra data {EXTRA} (used if present)")

[23:07:48] input.csv  /scratch/sguirale/Savant/input/input.csv


[23:07:48] extra data /scratch/sguirale/Savant/ML_model_v2/data/expanded_kd_ki.csv (used if present)


## 1. Load and combine

`input.csv` plus `expanded_kd_ki.csv` if it exists. Both are cleaned the same way before
being concatenated, so "which file a row came from" cannot become a hidden confound.

In [2]:
def load_and_clean(path, source_tag):
    if not path.exists():
        return None, None
    cols = ["seq", "smiles", "neg_log10_affinity_M"]
    have_kd = "affinity_Kd_nM" in pd.read_csv(path, nrows=0).columns
    if have_kd:
        cols.append("affinity_Kd_nM")
    raw = pd.read_csv(path, usecols=cols)
    n0 = len(raw)
    if have_kd:
        cap = np.zeros(len(raw), bool)
        for c in CENSOR_CAPS_NM:
            cap |= np.isclose(raw.affinity_Kd_nM.to_numpy(np.float64), c, rtol=1e-6)
    else:
        cap = np.zeros(len(raw), bool)          # get_more_data.py's output has no caps
    caps_raw = raw[cap][["seq", "smiles"]].copy()
    meas = raw[~cap]
    meas = meas[meas.neg_log10_affinity_M.between(MIN_PKD, MAX_PKD)].reset_index(drop=True)
    meas["source"] = source_tag
    log(f"  {source_tag}: {n0:,} rows -> {len(meas):,} measurements, "
        f"{int(cap.sum()):,} censor caps")
    return meas, caps_raw


meas_input, caps_input = load_and_clean(SRC, "input.csv")
meas_extra, caps_extra = load_and_clean(EXTRA, "expanded_kd_ki")

if meas_extra is None:
    log("no expanded_kd_ki.csv found - training on input.csv alone. Run "
        "scripts/get_more_data.py (chembl -> chembl-targets -> uniprot -> merge) to add "
        "the new ChEMBL/BindingDB-Ki data; this notebook does not require it.")
    meas_all = meas_input
else:
    log(f"combining input.csv ({len(meas_input):,}) with expanded_kd_ki.csv "
        f"({len(meas_extra):,})")
    meas_all = pd.concat([meas_input, meas_extra], ignore_index=True)
caps_all = pd.concat([c for c in (caps_input, caps_extra) if c is not None], ignore_index=True)

meas_all["li"] = pd.factorize(meas_all.smiles)[0]
meas_all["pi"] = pd.factorize(meas_all.seq)[0]
ligands  = meas_all.drop_duplicates("li").sort_values("li").smiles.to_numpy()
proteins = meas_all.drop_duplicates("pi").sort_values("pi").seq.to_numpy()
log(f"combined: {len(meas_all):,} measurement rows, {len(proteins):,} proteins, "
    f"{len(ligands):,} ligands")

lig_of = pd.Series(np.arange(len(ligands)), index=ligands)
pro_of = pd.Series(np.arange(len(proteins)), index=proteins)
caps_all = caps_all.assign(li=lig_of.reindex(caps_all.smiles).to_numpy(),
                           pi=pro_of.reindex(caps_all.seq).to_numpy()).dropna(subset=["li", "pi"])
caps_all["li"] = caps_all.li.astype(int); caps_all["pi"] = caps_all.pi.astype(int)
caps_all = caps_all.drop_duplicates(["li", "pi"])
log(f"censor-cap negatives mapped into the index: {len(caps_all):,}")

[23:07:58]   input.csv: 1,836,729 rows -> 1,689,437 measurements, 82,820 censor caps


[23:08:00]   expanded_kd_ki: 500,555 rows -> 462,163 measurements, 38,392 censor caps


[23:08:00] combining input.csv (1,689,437) with expanded_kd_ki.csv (462,163)


[23:08:01] combined: 2,151,600 measurement rows, 23,097 proteins, 1,168,247 ligands


[23:08:02] censor-cap negatives mapped into the index: 83,990


In [3]:
g = meas_all.groupby(["li", "pi"])["neg_log10_affinity_M"]
pairs = g.agg(pkd="median", n_meas="size", label_sd="std").reset_index()
pairs["label_sd"] = pairs.label_sd.fillna(0.0)
dup = pairs[pairs.n_meas > 1]
log(f"unique pairs {len(pairs):,} (collapsed {len(meas_all)-len(pairs):,} rows); "
    f"repeated {len(dup):,} ({100*len(dup)/len(pairs):.1f}%), "
    f"pooled sd {np.sqrt((dup.label_sd**2).mean()):.3f} pKd")
bad = pairs.label_sd > MAX_LABEL_SD
pairs = pairs[~bad].reset_index(drop=True)
pairs["w"] = np.sqrt(pairs.n_meas.to_numpy(np.float32))
caps = caps_all[~caps_all.set_index(["li", "pi"]).index.isin(pairs.set_index(["li", "pi"]).index)]
log(f"final: {len(pairs):,} pairs after dropping {int(bad.sum()):,} irreconcilable "
    f"(label_sd > {MAX_LABEL_SD}); {len(caps):,} capped negatives")
log(f"  binders (pKd >= {BINDER_PKD}): {100*(pairs.pkd >= BINDER_PKD).mean():.1f}%")
pairs.to_parquet(OUT / "pairs_clean.parquet", index=False)
del meas_all, meas_input, meas_extra, g; gc.collect()

[23:08:03] unique pairs 1,904,197 (collapsed 247,403 rows); repeated 174,251 (9.2%), pooled sd 0.763 pKd


[23:08:03] final: 1,892,952 pairs after dropping 11,245 irreconcilable (label_sd > 1.5); 76,184 capped negatives


[23:08:03]   binders (pKd >= 6.0): 69.4%


3

## 2. Static blocks - RDKit, ECFP4, AA composition, scaffolds

Unchanged from the earlier notebooks: computed once per unique entity, cached to
`work/static.npz`. New encoders and combined data mean this cannot reuse
`work_two_part/static.npz` directly (the ligand/protein index differs), so it rebuilds
here - but only once, then every later cell in this notebook reuses it.

In [4]:
from rdkit import Chem, RDLogger
from rdkit.Chem import (Crippen, Descriptors, Lipinski, rdFingerprintGenerator,
                        rdMolDescriptors)
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

def build_static():
    f = OUT / "static.npz"
    if f.exists():
        z = np.load(f, allow_pickle=True)
        if z["ecfp"].shape[0] == len(ligands) and z["aac"].shape[0] == len(proteins):
            log(f"reusing {f}")
            d = {k: z[k] for k in z.files}
            if "proteins" not in d or "ligands" not in d:
                ## Earlier saves of this file did not persist the protein/ligand order,
                ## which is exactly what a SLURM-array embedding manifest for
                ## embed_structure_encoders.py needs to line up against - without it,
                ## there is no way to reconstruct which embedding row belongs to which
                ## protein. Both arrays are already sitting in memory (computed from
                ## meas_all before this function runs), so this costs nothing to add -
                ## no RDKit recompute needed, just rewriting the same file.
                log("  static.npz predates saving proteins/ligands - adding them now "
                    "(no RDKit recompute, already in memory)")
                d["proteins"], d["ligands"] = proteins, ligands
                np.savez_compressed(f, **d)
            return d, json.loads((OUT / "rdkit_columns.json").read_text())
        log(f"{f.name} shape mismatch (data changed) - rebuilding")

    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=ECFP_BITS)
    d11, full, fps, scaf, ok = [], [], [], [], []
    for i, s in enumerate(ligands):
        m = Chem.MolFromSmiles(s); ok.append(m is not None)
        if m is None:
            d11.append([0.0]*11); full.append({}); fps.append(np.zeros(ECFP_BITS)); scaf.append("")
            continue
        mh = Chem.AddHs(m)
        d11.append([Descriptors.MolWt(mh), Crippen.MolLogP(mh), Lipinski.NumHDonors(mh),
                    Lipinski.NumHAcceptors(mh), Lipinski.NumRotatableBonds(mh),
                    rdMolDescriptors.CalcTPSA(mh), rdMolDescriptors.CalcNumRings(mh),
                    rdMolDescriptors.CalcNumAromaticRings(mh), mh.GetNumAtoms(),
                    mh.GetNumHeavyAtoms(), Chem.GetFormalCharge(mh)])
        full.append(Descriptors.CalcMolDescriptors(m))
        fps.append(gen.GetCountFingerprintAsNumPy(m))
        sc = MurckoScaffold.MurckoScaffoldSmiles(mol=m); scaf.append(sc if sc else Chem.MolToSmiles(m))
        if (i+1) % 100_000 == 0: log(f"  ligands {i+1:,}/{len(ligands):,}")
    fd = pd.DataFrame(full).astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    fd = fd.loc[:, fd.max() != fd.min()]
    aac = np.array([[s.count(a)/len(s) for a in AA] + [len(s)] for s in proteins], np.float32)

    d = dict(d11=np.asarray(d11, np.float32), dfull=fd.to_numpy(np.float32),
             ecfp=np.asarray(fps, np.float32), aac=aac, scaf=np.asarray(scaf),
             lig_ok=np.asarray(ok), proteins=proteins, ligands=ligands)
    np.savez_compressed(f, **d)
    (OUT / "rdkit_columns.json").write_text(json.dumps(list(fd.columns)))
    return d, list(fd.columns)

S, RD_COLS = build_static()
pairs = pairs[S["lig_ok"][pairs.li.to_numpy()]].reset_index(drop=True)
caps  = caps[S["lig_ok"][caps.li.to_numpy()]].reset_index(drop=True)
log(f"d11 {S['d11'].shape}  rdkit {S['dfull'].shape}  ecfp {S['ecfp'].shape}  aa {S['aac'].shape}")
log(f"{len(pairs):,} pairs, {len(caps):,} caps after dropping unparseable ligands")

[23:08:19] static.npz shape mismatch (data changed) - rebuilding


[23:31:14]   ligands 100,000/1,168,247


[23:54:30]   ligands 200,000/1,168,247


[00:18:25]   ligands 300,000/1,168,247


[00:42:24]   ligands 400,000/1,168,247


[01:07:30]   ligands 500,000/1,168,247


... (7 outputs left out)

[04:05:24]   ligands 1,100,000/1,168,247


/apps/pkg/conda/envs/pytorch-2.9.1/lib/python3.13/site-packages/pandas/core/dtypes/astype.py:136: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=copy)


[04:28:27] d11 (1168247, 11)  rdkit (1168247, 215)  ecfp (1168247, 1024)  aa (23097, 21)


[04:28:27] 1,892,947 pairs, 76,184 caps after dropping unparseable ligands


## 3. Encoders

esmc-300m and ChemBERTa-77M-MLM, fixed. Cached per unique entity, so this cell is a no-op
on any re-run.

In [5]:
def embed_texts(texts, hf, batch_size, max_length):
    import torch
    from transformers import AutoModel, AutoTokenizer
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"  {hf} on {dev}, {len(texts):,} items")
    mdl = AutoModel.from_pretrained(hf, trust_remote_code=True).to(dev).eval()
    try:
        tok = AutoTokenizer.from_pretrained(hf, trust_remote_code=True)
    except Exception:
        tok = getattr(mdl, "tokenizer", None)
        if tok is None: raise
    order = np.argsort([len(t) for t in texts]); out = None
    with torch.inference_mode():
        for st in range(0, len(order), batch_size):
            sel = order[st:st+batch_size]
            b = tok([texts[i] for i in sel], padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt")
            b = {k: v.to(dev) for k, v in b.items()}
            h = mdl(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            p = ((h*m).sum(1)/m.sum(1).clamp_min(1)).float().cpu().numpy()
            if out is None: out = np.zeros((len(texts), p.shape[1]), np.float32)
            out[sel] = p
            if (st // batch_size) % 200 == 0:
                log(f"    {min(st+batch_size, len(order)):,}/{len(order):,}")
    del mdl
    if dev == "cuda":
        import torch; torch.cuda.empty_cache()
    return out


def cached(enc, texts, batch_size, max_length):
    '''Embed with a cache that survives the corpus SHRINKING.

    Each cache carries a companion `<slug>_keys.npy` recording which text produced which
    row. That buys two things a bare row-count check cannot:

      * when rows are only REMOVED (e.g. the benchmark-gate filter in get_more_data.py
        drops a few hundred ligands), the surviving rows are re-indexed out of the old
        matrix instead of re-embedding the other ~1.17M from scratch. That is the
        difference between a CPU-only re-run and one that needs a GPU.
      * a row count that matches by coincidence while the CONTENT changed is caught,
        rather than silently pairing every ligand with another ligand's embedding.

    Requires texts to be unique, which `ligands`/`proteins` are by construction
    (drop_duplicates on the factorised index) - asserted rather than assumed.
    '''
    p = EMB / f"{enc['slug']}.npy"
    kp = EMB / f"{enc['slug']}_keys.npy"
    texts = list(texts)
    assert len(set(texts)) == len(texts), f"{enc['slug']}: texts must be unique"

    def close(mm_owner):
        '''Release a memmap's file handle.

        Necessary because np.asarray() on a float32 memmap returns the MEMMAP ITSELF,
        not a copy - so returning it leaves the file open for the life of the notebook,
        and any later write to that same cache fails (OSError on Windows; on Linux it
        truncates a file that is still mapped). Every path below materialises with
        np.array(...) and then closes, rather than trusting refcounts.
        '''
        mm = getattr(mm_owner, "_mmap", None)
        if mm is not None:
            mm.close()

    if p.exists():
        a = np.load(p, mmap_mode="r")
        old = np.load(kp, allow_pickle=True) if kp.exists() else None
        rebuild_reason = None

        if old is None:
            if a.shape[0] == len(texts):
                log(f"  reuse {p.name} {a.shape} (no key file - matched on row count only)")
                out = np.array(a, dtype=np.float32)
                close(a)
                np.save(kp, np.asarray(texts, object))
                log(f"  wrote {kp.name} so a future corpus change can subset instead of "
                    f"re-embedding")
                return out
            rebuild_reason = (f"{a.shape[0]:,} rows, expected {len(texts):,}, and no key "
                              f"file to map them onto")
        else:
            pos = pd.Series(np.arange(len(old)), index=old)
            idx = pos.reindex(texts).to_numpy()
            missing = int(np.isnan(idx).sum())
            if missing == 0:
                if (len(texts) == a.shape[0]
                        and np.array_equal(np.asarray(old), np.asarray(texts, object))):
                    log(f"  reuse {p.name} {a.shape}")
                    out = np.array(a, dtype=np.float32)
                    close(a)
                    return out
                log(f"  {p.name}: {a.shape[0]:,} -> {len(texts):,} rows, every text already "
                    f"in the cache - SUBSETTING, no re-embedding needed")
                sub = np.array(a[idx.astype(int)], dtype=np.float32)
                close(a)
                np.save(p, sub); np.save(kp, np.asarray(texts, object))
                return sub
            rebuild_reason = f"{missing:,} of {len(texts):,} texts are not in the cache"

        close(a)
        log(f"  {p.name}: {rebuild_reason} - REBUILDING (this is the step that needs a GPU)")

    arr = embed_texts(texts, enc["hf"], batch_size, max_length)
    np.save(p, arr); np.save(kp, np.asarray(texts, object))
    return arr

P_EMB = cached(PROTEIN_ENC, list(proteins), 8, 1024)
L_EMB = cached(LIGAND_ENC, list(ligands), 512, 256)
log(f"protein {P_EMB.shape}  ligand {L_EMB.shape}")

[04:28:27]   reuse esmc-300m.npy (23097, 960)


[04:28:29]   chemberta-mlm77.npy: 1,168,694 -> 1,168,247 rows, every text already in the cache - SUBSETTING, no re-embedding needed


[04:28:37] protein (23097, 960)  ligand (1168247, 384)


## 4. Physicochemical complementarity — new

RDKit descriptors and ECFP4 describe the ligand in isolation; AA composition describes the
protein in isolation. Neither says whether the two are a *fit*. These features are cheap,
sequence/SMILES-only proxies for the interaction terms `savant_v2/savant2/interactions.py`
computes from an actual docked structure - deliberately coarser (composition, not
geometry), because the entire point is not needing a pose.

Five complementarity axes, each as **both** the raw protein-side fraction (so the model can
use it alone) and the explicit product with the matching ligand scalar (so the model does
not have to discover the interaction through tree splits - handed the cross-term directly):

| axis | ligand side | protein side (residue fraction) |
|---|---|---|
| charge (+) | max(formal_charge, 0) | D + E (acidic) |
| charge (-) | max(-formal_charge, 0) | K + R + H (basic) |
| H-bond donor -> acceptor | NumHDonors | S,T,Y,N,Q,D,E (HBA-capable side chains) |
| H-bond acceptor -> donor | NumHAcceptors | S,T,Y,N,Q,K,R,H,W (HBD-capable side chains) |
| hydrophobic | LogP | A,V,L,I,M,F,W,Y,C (hydrophobic) |
| aromatic / pi-stacking | NumAromaticRings | F,Y,W,H (aromatic side chains) |
| size | log(HeavyAtoms+1) | log(protein length) |

Plus a **Hadamard block**: both embeddings PCA-projected to the same width (fit on the
training fold only) and multiplied elementwise - a direct, cheap interaction term at the
representation level, not just the hand-engineered scalar level.

In [6]:
HYDRO = set("AVLIMFWYC")
AROM_P = set("FYWH")
ACID_P = set("DE")
BASIC_P = set("KRH")
HBA_P = set("STYNQDE")
HBD_P = set("STYNQKRHW")

def protein_fracs(seqs):
    n = np.array([len(s) for s in seqs], np.float32)
    def frac(charset):
        return np.array([sum(s.count(c) for c in charset) for s in seqs], np.float32) / n
    return dict(acidic=frac(ACID_P), basic=frac(BASIC_P), hba=frac(HBA_P),
               hbd=frac(HBD_P), hydro=frac(HYDRO), arom=frac(AROM_P),
               log_len=np.log1p(n))

PF = protein_fracs(proteins)

def ligand_scalars(smiles):
    charge, hbd, hba, logp, arom, heavy = [], [], [], [], [], []
    for s in smiles:
        m = Chem.MolFromSmiles(s)
        if m is None:
            charge.append(0.0); hbd.append(0.0); hba.append(0.0)
            logp.append(0.0); arom.append(0.0); heavy.append(0.0)
            continue
        charge.append(float(Chem.GetFormalCharge(m)))
        hbd.append(float(Lipinski.NumHDonors(m))); hba.append(float(Lipinski.NumHAcceptors(m)))
        logp.append(float(Crippen.MolLogP(m))); arom.append(float(rdMolDescriptors.CalcNumAromaticRings(m)))
        heavy.append(float(m.GetNumHeavyAtoms()))
    return dict(charge=np.array(charge, np.float32), hbd=np.array(hbd, np.float32),
               hba=np.array(hba, np.float32), logp=np.array(logp, np.float32),
               arom=np.array(arom, np.float32), log_heavy=np.log1p(np.array(heavy, np.float32)))

LS = ligand_scalars(ligands)
log(f"ligand scalar block ready for {len(ligands):,} ligands; "
    f"protein fraction block ready for {len(proteins):,} proteins")


def complementarity_block(li, pi):
    '''(n, 14) float32: 7 raw protein fractions/products, matched to the pair's ligand.'''
    lig_pos = np.maximum(LS["charge"][li], 0); lig_neg = np.maximum(-LS["charge"][li], 0)
    cols = [
        PF["acidic"][pi], lig_pos * PF["acidic"][pi],
        PF["basic"][pi],  lig_neg * PF["basic"][pi],
        PF["hba"][pi],    LS["hbd"][li] * PF["hba"][pi],
        PF["hbd"][pi],    LS["hba"][li] * PF["hbd"][pi],
        PF["hydro"][pi],  LS["logp"][li] * PF["hydro"][pi],
        PF["arom"][pi],   LS["arom"][li] * PF["arom"][pi],
        PF["log_len"][pi], LS["log_heavy"][li] * PF["log_len"][pi],
    ]
    return np.stack(cols, axis=1).astype(np.float32)

COMPLEMENT_NAMES = ["prot_acidic", "charge_pos_x_acidic", "prot_basic", "charge_neg_x_basic",
                    "prot_hba", "lig_hbd_x_prot_hba", "prot_hbd", "lig_hba_x_prot_hbd",
                    "prot_hydro", "lig_logp_x_hydro", "prot_arom", "lig_arom_x_prot_arom",
                    "prot_log_len", "lig_log_heavy_x_log_len"]
_test = complementarity_block(pairs.li.to_numpy()[:100], pairs.pi.to_numpy()[:100])
assert _test.shape == (100, 14) and np.isfinite(_test).all()
log(f"complementarity block: {_test.shape[1]} features per pair, verified finite")

[04:40:01] ligand scalar block ready for 1,168,247 ligands; protein fraction block ready for 23,097 proteins


[04:40:01] complementarity block: 14 features per pair, verified finite


## 5. The split — protein and scaffold double-disjoint, kept as a 2x2

Exactly the methodology this project's earlier notebooks established is necessary: a
plain protein-grouped split scored 0.57 internally while the true screening condition
scored 0.33, because it let chemistry leak freely. Four cells, all disjoint from training
on at least one axis, none discarded.

In [7]:
rng = np.random.default_rng(SEED)
pi_all = pairs.pi.to_numpy(); li_all = pairs.li.to_numpy()
y_all  = pairs.pkd.to_numpy(np.float32); w_all = pairs.w.to_numpy(np.float32)
scaf_u = pd.factorize(S["scaf"])[0]; sp_all = scaf_u[li_all]

per_p = pairs.groupby("pi").size(); bins = np.floor(np.log2(per_p.clip(lower=1))).astype(int)
test_p = []
for b in np.unique(bins):
    ps = per_p.index[bins == b].to_numpy()
    test_p.extend(rng.choice(ps, size=max(1, int(round(TEST_PROTEIN_FRAC*len(ps)))),
                             replace=False).tolist())
test_p = set(test_p)
uscaf = np.unique(sp_all)
test_s = set(rng.choice(uscaf, size=max(1, int(round(TEST_SCAFFOLD_FRAC*len(uscaf)))),
                        replace=False).tolist())

npr = np.array([p in test_p for p in pi_all]); ns_ = np.array([s in test_s for s in sp_all])
train_pool = np.where(~npr & ~ns_)[0]; rng.shuffle(train_pool)
n_ind = int(INDOMAIN_FRAC * len(train_pool))
CELLS = {"seen prot / seen chem":   train_pool[:n_ind],
         "seen prot / NOVEL chem":  np.where(~npr & ns_)[0],
         "NOVEL prot / seen chem":  np.where(npr & ~ns_)[0],
         "NOVEL prot / NOVEL chem": np.where(npr & ns_)[0]}
tr_idx = train_pool[n_ind:]
DEPLOY = "NOVEL prot / NOVEL chem"

log(f"train {len(tr_idx):,} pairs / {len(np.unique(pi_all[tr_idx])):,} proteins")
for k, v in CELLS.items():
    log(f"  {k:24s} {len(v):>9,} pairs / {len(np.unique(pi_all[v])):>6,} proteins")
tr_prot, tr_scaf = set(pi_all[tr_idx]), set(sp_all[tr_idx])
for k in ("NOVEL prot / seen chem", "NOVEL prot / NOVEL chem"):
    assert not (set(pi_all[CELLS[k]]) & tr_prot), f"protein leak into {k}"
for k in ("seen prot / NOVEL chem", "NOVEL prot / NOVEL chem"):
    assert not (set(sp_all[CELLS[k]]) & tr_scaf), f"scaffold leak into {k}"
log("no protein or scaffold leakage into the cells that claim to hold them out")

cap_pi, cap_li = caps.pi.to_numpy(), caps.li.to_numpy(); cap_sp = scaf_u[cap_li]
cap_np_ = np.array([p in test_p for p in cap_pi]); cap_ns = np.array([s in test_s for s in cap_sp])
CAP_TRAIN = np.where(~cap_np_ & ~cap_ns)[0]
CAP_DEPLOY = np.where(cap_np_ & cap_ns)[0]
log(f"capped negatives: train {len(CAP_TRAIN):,}, deployment cell {len(CAP_DEPLOY):,}")

[04:40:04] train 1,146,448 pairs / 16,357 proteins


[04:40:04]   seen prot / seen chem       60,339 pairs /  4,377 proteins


[04:40:04]   seen prot / NOVEL chem     288,782 pairs /  7,132 proteins


[04:40:04]   NOVEL prot / seen chem     321,796 pairs /  4,188 proteins


[04:40:04]   NOVEL prot / NOVEL chem     75,582 pairs /  1,812 proteins


[04:40:04] no protein or scaffold leakage into the cells that claim to hold them out


[04:40:04] capped negatives: train 48,448, deployment cell 3,111


## 6. Stage 1 — binder classifier

Positives: pKd >= 6. Negatives: measured pairs pKd < 6, plus the censor-cap rows (the
only genuine non-binders in the corpus). Threshold tuned to a target recall on a held-out
fold, not left at 0.5 - the specific defect that made v1's gate reject 288 measured
binders on the benchmark.

In [8]:
import xgboost as xgb

def assemble(idx, kind="pair"):
    '''(n, width) float32: ligand_emb, d11, rdkit, ecfp, protein_emb, aa, complementarity.'''
    if kind == "pair":
        li, pi = li_all[idx], pi_all[idx]
    else:                                    # "cap"
        li, pi = cap_li[idx], cap_pi[idx]
    comp = complementarity_block(li, pi)
    return np.hstack([L_EMB[li], S["d11"][li], S["dfull"][li], S["ecfp"][li],
                      P_EMB[pi], S["aac"][pi], comp]).astype(np.float32)


N_FEATURES = assemble(tr_idx[:5]).shape[1]
log(f"feature width: {N_FEATURES} "
    f"(ligand {L_EMB.shape[1]} + d11 11 + rdkit {S['dfull'].shape[1]} + ecfp {ECFP_BITS} + "
    f"protein {P_EMB.shape[1]} + aa 21 + complementarity 14)")

XGB = dict(max_depth=8, eta=0.05, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=10, reg_lambda=2.0, tree_method="hist",
          nthread=N_THREADS, seed=SEED)
TARGET_RECALL = 0.95

## Round budgets. The previous full run ended at best_iteration 3996 of 4000 with
## val-RMSE STILL falling (1.19609 at the last round) - the regressor was round-limited,
## not converged, so the cap was leaving free performance on the table. Raised well past
## where it should plateau; early stopping decides the real endpoint, so a high cap costs
## nothing except when it is actually used.
##
## Runtime: the 4,000-round run took ~2h50m (~2.5 s/round) on ~1M x 2,629 features. At
## 20,000 the worst case is ~14h if it never early-stops. Lower N_ROUNDS_REG if that does
## not fit your queue - it is a cap, not a target.
N_ROUNDS_REG = 20_000
N_ROUNDS_CLF = 3_000
N_ROUNDS_RANK = 2_000
## More patience than the old 50: late gains are small and noisy (~0.001 RMSE per 200
## rounds near the end), and a 50-round window can stop on noise before a real plateau.
EARLY_STOP = 200

## inner validation fold, held out by protein like the outer split
tr_p = np.unique(pi_all[tr_idx])
val_p = set(rng.choice(tr_p, size=max(1, int(0.15*len(tr_p))), replace=False).tolist())
in_val = np.array([p in val_p for p in pi_all[tr_idx]])
fit_idx, val_idx = tr_idx[~in_val], tr_idx[in_val]
cap_in_val = np.array([p in val_p for p in cap_pi[CAP_TRAIN]])
cap_fit, cap_val = CAP_TRAIN[~cap_in_val], CAP_TRAIN[cap_in_val]
log(f"fit {len(fit_idx):,} pairs + {len(cap_fit):,} caps | "
    f"val {len(val_idx):,} pairs + {len(cap_val):,} caps")


def clf_xy(idx, cidx):
    Xs = [assemble(idx, "pair")]
    ys = [(y_all[idx] >= BINDER_PKD).astype(np.int8)]
    ws = [w_all[idx]]
    if USE_CAPS_AS_NEGATIVES and len(cidx):
        Xs.append(assemble(cidx, "cap"))
        ys.append(np.zeros(len(cidx), np.int8))
        ws.append(np.full(len(cidx), float(np.median(w_all[idx])), np.float32))
    return np.vstack(Xs), np.concatenate(ys), np.concatenate(ws)


Xf, yf, wf = clf_xy(fit_idx, cap_fit)
Xv, yv, wv = clf_xy(val_idx, cap_val)
log(f"stage-1 train {Xf.shape}, positive rate {yf.mean():.3f}")
dtr = xgb.QuantileDMatrix(Xf, label=yf, weight=wf); del Xf; gc.collect()
dva = xgb.QuantileDMatrix(Xv, label=yv, ref=dtr)
clf = xgb.train({**XGB, "objective": "binary:logistic", "eval_metric": "aucpr"},
                dtr, num_boost_round=N_ROUNDS_CLF, evals=[(dva, "val")],
                early_stopping_rounds=EARLY_STOP, verbose_eval=200)
pv = clf.inplace_predict(Xv)
from sklearn.metrics import average_precision_score, roc_auc_score
THRESH = float(np.quantile(pv[yv == 1], 1 - TARGET_RECALL))
log(f"stage-1 val ROC {roc_auc_score(yv, pv):.4f}  PR {average_precision_score(yv, pv):.4f}  "
    f"threshold@{TARGET_RECALL:.0%} recall = {THRESH:.4f}")
del Xv, dtr, dva; gc.collect()

[04:40:09] feature width: 2629 (ligand 384 + d11 11 + rdkit 215 + ecfp 1024 + protein 960 + aa 21 + complementarity 14)


[04:40:10] fit 1,015,808 pairs + 42,459 caps | val 130,640 pairs + 5,989 caps


[04:40:22] stage-1 train (1058267, 2629), positive rate 0.665


[0]	val-aucpr:0.73602


[200]	val-aucpr:0.81467


[400]	val-aucpr:0.82886


... (11 outputs left out)

[2800]	val-aucpr:0.85969


[2999]	val-aucpr:0.86014


[07:11:54] stage-1 val ROC 0.8014  PR 0.8601  threshold@95% recall = 0.2818


66

## 7. Stage 2 — regressor, two objectives on identical inputs

Trained on **every** measured pair (pKd in [4, 12]), not binders-only - absolute pKd
across the full range is the primary deliverable this notebook targets.

- `reg_squared`: `reg:squarederror`, optimises absolute pKd directly.
- `reg_ranking`: `rank:pairwise` grouped by protein (`qid`), optimises within-target
  order directly. Requires rows sorted by group, which is why the fit/val index arrays
  are re-sorted by `pi` before building the ranking DMatrix - XGBoost's `qid` interface
  assumes contiguous groups and silently mis-groups an unsorted input instead of erroring,
  so this is checked with an assertion rather than trusted.

In [9]:
Xtr = assemble(fit_idx, "pair")
dtr = xgb.QuantileDMatrix(Xtr, label=y_all[fit_idx], weight=w_all[fit_idx]); del Xtr; gc.collect()
Xva = assemble(val_idx, "pair")
dva = xgb.QuantileDMatrix(Xva, label=y_all[val_idx], ref=dtr); del Xva; gc.collect()
reg_squared = xgb.train({**XGB, "objective": "reg:squarederror", "eval_metric": "rmse"},
                        dtr, num_boost_round=N_ROUNDS_REG, evals=[(dva, "val")],
                        early_stopping_rounds=EARLY_STOP, verbose_eval=500)
del dtr, dva; gc.collect()
log(f"reg_squared best_iteration {getattr(reg_squared, 'best_iteration', -1)}")

[0]	val-rmse:1.39523


[500]	val-rmse:1.19360


[1000]	val-rmse:1.17331


[1500]	val-rmse:1.16133


[2000]	val-rmse:1.15284


[2500]	val-rmse:1.14686


... (4 outputs left out)

[5000]	val-rmse:1.13416


[5500]	val-rmse:1.13328


[5549]	val-rmse:1.13326


[11:57:08] reg_squared best_iteration 5349


In [10]:
def sort_by_group(idx):
    order = np.argsort(pi_all[idx], kind="stable")
    return idx[order]

fit_sorted, val_sorted = sort_by_group(fit_idx), sort_by_group(val_idx)
## qid must be non-decreasing within the matrix - this is the assumption XGBoost makes
## silently; checked here instead of trusted, because a violation degrades the ranking
## loss without ever raising an error.
assert (np.diff(pi_all[fit_sorted]) >= 0).all(), "fit rows not sorted by group"
assert (np.diff(pi_all[val_sorted]) >= 0).all(), "val rows not sorted by group"

def group_sizes(idx):
    _, counts = np.unique(pi_all[idx], return_counts=True)
    return counts

## No `weight=` here - caught by actually running this. Once set_group() marks a
## DMatrix as grouped, XGBoost reinterprets `weight` as one value PER GROUP (per
## protein), not per row - passing the per-row replicate-count weight (length =
## n_rows) against ~9k groups fails with a hard size-mismatch error, not a silent
## misuse. Re-weighting by replicate count within a ranking group has no clean
## equivalent in XGBoost's classic ranking API, so it is dropped for this objective
## only - reg_squared keeps it. Every pair within a group is weighted equally here.
Xtr = assemble(fit_sorted, "pair")
dtr = xgb.DMatrix(Xtr, label=y_all[fit_sorted]); del Xtr; gc.collect()
dtr.set_group(group_sizes(fit_sorted))
Xva = assemble(val_sorted, "pair")
dva = xgb.DMatrix(Xva, label=y_all[val_sorted]); del Xva; gc.collect()
dva.set_group(group_sizes(val_sorted))

## eval_metric="ndcg" fails here, caught by actually running it: XGBoost's NDCG assumes
## graded relevance (small non-negative INTEGER grades, e.g. 0-4 stars), and errors with
## "label must be either 0 or positive integer" the moment it sees continuous pKd values.
## "auc" is XGBoost's pairwise-concordance metric for ranking mode - it accepts any
## real-valued label (it only needs to compare two labels' order) and is the more
## appropriate metric for continuous affinity anyway: NDCG's discounting scheme is built
## for categorical relevance judgments, not a real-valued target.
reg_ranking = xgb.train({**XGB, "objective": "rank:pairwise", "eval_metric": "auc"},
                        dtr, num_boost_round=N_ROUNDS_RANK, evals=[(dva, "val")],
                        early_stopping_rounds=EARLY_STOP, verbose_eval=200)
del dtr, dva; gc.collect()
log(f"reg_ranking best_iteration {getattr(reg_ranking, 'best_iteration', -1)}")

[0]	val-auc:0.53950


[200]	val-auc:0.57944


[400]	val-auc:0.58939


[549]	val-auc:0.59189


[12:33:04] reg_ranking best_iteration 349


## 8. Evaluate — both objectives, all four cells, with and without the classifier gate

`rank:pairwise` scores are a relative order, not calibrated pKd - `pred_std_ratio` and
RMSE are not meaningful for it and are left as NaN rather than a misleading number.
Pearson/Spearman still make sense since XGBoost's ranking score is monotonic with rank.

In [11]:
from scipy.stats import spearmanr

def decompose(y, p, g):
    d = pd.DataFrame({"y": y, "p": p, "g": g})
    m = d.groupby("g")[["y", "p"]].mean()
    between = float(np.corrcoef(m.y, m.p)[0, 1]) if len(m) > 2 else np.nan
    c = d.join(m, on="g", rsuffix="_m"); cy, cp = c.y - c.y_m, c.p - c.p_m
    within = float(np.corrcoef(cy, cp)[0, 1]) if cy.std() > 1e-9 and cp.std() > 1e-9 else np.nan
    return between, within

def report(y, p, g, is_ranking, **extra):
    b, w = decompose(y, p, g)
    row = {"n": len(y), "pearson": float(np.corrcoef(y, p)[0, 1]),
          "spearman": float(spearmanr(y, p).correlation),
          "between_r": b, "within_r": w, **extra}
    if not is_ranking:
        row["rmse"] = float(np.sqrt(((y-p)**2).mean()))
        row["rmse_const"] = float(y.std())
        row["pred_std_ratio"] = float(p.std()/y.std())
    return row


rows = []
for cell, idx in CELLS.items():
    X = assemble(idx, "pair")
    p_bind = clf.inplace_predict(X)
    p_sq = reg_squared.inplace_predict(X)
    d_rank = xgb.DMatrix(X); p_rank = reg_ranking.predict(d_rank)
    del X, d_rank; gc.collect()
    y, g = y_all[idx], pi_all[idx]
    gated_sq = np.where(p_bind >= THRESH, p_sq, p_sq.min() - 1.0)

    rows.append({"cell": cell, "model": "reg_squared", **report(y, p_sq, g, False)})
    rows.append({"cell": cell, "model": "reg_ranking", **report(y, p_rank, g, True)})
    rows.append({"cell": cell, "model": "reg_squared + gate", **report(y, gated_sq, g, False)})

    lab = (y >= BINDER_PKD).astype(int)
    if 0 < lab.sum() < len(lab):
        rows.append({"cell": cell, "model": "classifier",
                     "n": len(y), "roc_auc": roc_auc_score(lab, p_bind),
                     "pr_auc": average_precision_score(lab, p_bind)})
    log(f"{cell}: scored")

R = pd.DataFrame(rows).set_index(["cell", "model"])
R.to_csv(OUT / "internal_2x2.csv")
display(R.round(4))
print("\nPearson by cell x model:")
display(R.pearson.unstack("model").reindex(list(CELLS)).round(4))

[12:33:11] seen prot / seen chem: scored


[12:33:40] seen prot / NOVEL chem: scored


[12:34:13] NOVEL prot / seen chem: scored


[12:34:21] NOVEL prot / NOVEL chem: scored


n  pearson  spearman  \
cell                    model                                           
seen prot / seen chem   reg_squared          60339   0.8578    0.8522   
                        reg_ranking          60339   0.3645    0.3507   
                        reg_squared + gate   60339   0.7759    0.8463   
                        classifier           60339      NaN       NaN   
seen prot / NOVEL chem  reg_squared         288782   0.7903    0.7800   
                        reg_ranking         288782   0.3257    0.3197   
                        reg_squared + gate  288782   0.7155    0.7758   
                        classifier          288782      NaN       NaN   
NOVEL prot / seen chem  reg_squared         321796   0.5676    0.5554   
                        reg_ranking         321796   0.3166    0.3072   
                        reg_squared + gate  321796   0.5204    0.5542   
                        classifier          321796      NaN       NaN   
NOVEL prot / NOVEL chem reg_squared          75582   0.5266    0.5130   
                        reg_ranking          75582   0.2906    0.2879   
                        reg_squared + gate   75582   0.4875    0.5127   
                        classifier           75582      NaN       NaN   

                                            between_r  within_r    rmse  \
cell                    model                                             
seen prot / seen chem   reg_squared            0.8266    0.8119  0.7247   
                        reg_ranking            0.3761    0.3783     NaN   
                        reg_squared + gate     0.7627    0.6972  1.4178   
                        classifier                NaN       NaN     NaN   
seen prot / NOVEL chem  reg_squared            0.7611    0.7078  0.8612   
                        reg_ranking            0.3359    0.3323     NaN   
                        reg_squared + gate     0.6896    0.5964  1.4435   
                        classifier                NaN       NaN     NaN   
NOVEL prot / seen chem  reg_squared            0.7034    0.4703  1.1677   
                        reg_ranking            0.3109    0.3005     NaN   
                        reg_squared + gate     0.6373    0.3877  1.4905   
                        classifier                NaN       NaN     NaN   
NOVEL prot / NOVEL chem reg_squared            0.6737    0.3991  1.2147   
                        reg_ranking            0.2997    0.2694     NaN   
                        reg_squared + gate     0.6186    0.3262  1.4714   
                        classifier                NaN       NaN     NaN   

                                            rmse_const  pred_std_ratio  \
cell                    model                                            
seen prot / seen chem   reg_squared             1.4044          0.8116   
                        reg_ranking                NaN             NaN   
                        reg_squared + gate      1.4044          1.4583   
                        classifier                 NaN             NaN   
seen prot / NOVEL chem  reg_squared             1.4012          0.7562   
                        reg_ranking                NaN             NaN   
                        reg_squared + gate      1.4012          1.3710   
                        classifier                 NaN             NaN   
NOVEL prot / seen chem  reg_squared             1.3906          0.5407   
                        reg_ranking                NaN             NaN   
                        reg_squared + gate      1.3906          1.0207   
                        classifier                 NaN             NaN   
NOVEL prot / NOVEL chem reg_squared             1.3956          0.5089   
                        reg_ranking                NaN             NaN   
                        reg_squared + gate      1.3956          0.9328   
                        classifier                 NaN             NaN   

                                            roc_au


Pearson by cell x model:


model,classifier,reg_ranking,reg_squared,reg_squared + gate
cell,,,,
seen prot / seen chem,NaN,0.3645,0.8578,0.7759
seen prot / NOVEL chem,NaN,0.3257,0.7903,0.7155
NOVEL prot / seen chem,NaN,0.3166,0.5676,0.5204
NOVEL prot / NOVEL chem,NaN,0.2906,0.5266,0.4875


## 9. External benchmark

Scored so the numbers sit beside `bench_summary.md` (Boltz-2 0.412, PRODIGY-LIG 0.333,
Savant esm8 0.330, PLAPT 0.021, constant-mean RMSE 1.273) and the earlier per-target
screening table (Savant-v2 esm650 0.311 mean per-target Spearman vs Boltz-2's 0.228).
Both the pooled number and the per-target one are reported, because this project's own
finding is that the pooled number on this particular benchmark is dominated by an
adversarial between-target axis - report one without the other and the result is
misleading in whichever direction gets omitted.

In [12]:
if not BENCH.exists():
    log(f"no benchmark at {BENCH} - skipping")
else:
    bm = pd.read_csv(BENCH, usecols=["target_sequence", "ligand_smiles",
                                     "neg_log10_affinity_M", "uniprot_id"]).dropna()
    keep = [Chem.MolFromSmiles(s) is not None for s in bm.ligand_smiles]
    bm = bm[keep].reset_index(drop=True)
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=ECFP_BITS)
    d11b, fullb, fpb = [], [], []
    for s in bm.ligand_smiles:
        m = Chem.MolFromSmiles(s); mh = Chem.AddHs(m)
        d11b.append([Descriptors.MolWt(mh), Crippen.MolLogP(mh), Lipinski.NumHDonors(mh),
                     Lipinski.NumHAcceptors(mh), Lipinski.NumRotatableBonds(mh),
                     rdMolDescriptors.CalcTPSA(mh), rdMolDescriptors.CalcNumRings(mh),
                     rdMolDescriptors.CalcNumAromaticRings(mh), mh.GetNumAtoms(),
                     mh.GetNumHeavyAtoms(), Chem.GetFormalCharge(mh)])
        fullb.append(Descriptors.CalcMolDescriptors(m))
        fpb.append(gen.GetCountFingerprintAsNumPy(m))
    fdb = (pd.DataFrame(fullb).astype(np.float32).replace([np.inf, -np.inf], np.nan)
             .fillna(0.0).reindex(columns=RD_COLS, fill_value=0.0).to_numpy(np.float32))
    Lb = embed_texts(list(bm.ligand_smiles), LIGAND_ENC["hf"], 256, 256)
    Pb = embed_texts(list(bm.target_sequence), PROTEIN_ENC["hf"], 8, 1024)
    aab = np.array([[s.count(a)/len(s) for a in AA] + [len(s)]
                    for s in bm.target_sequence], np.float32)
    bpf = protein_fracs(list(bm.target_sequence))
    bls = ligand_scalars(list(bm.ligand_smiles))
    bcomp = np.stack([
        bpf["acidic"], np.maximum(bls["charge"],0)*bpf["acidic"],
        bpf["basic"],  np.maximum(-bls["charge"],0)*bpf["basic"],
        bpf["hba"],    bls["hbd"]*bpf["hba"],
        bpf["hbd"],    bls["hba"]*bpf["hbd"],
        bpf["hydro"],  bls["logp"]*bpf["hydro"],
        bpf["arom"],   bls["arom"]*bpf["arom"],
        bpf["log_len"], bls["log_heavy"]*bpf["log_len"],
    ], axis=1).astype(np.float32)
    Xb = np.hstack([Lb, np.asarray(d11b, np.float32), fdb, np.asarray(fpb, np.float32),
                    Pb, aab, bcomp]).astype(np.float32)

    yb = bm.neg_log10_affinity_M.to_numpy(np.float32); gb = bm.uniprot_id.to_numpy()
    p_sq = reg_squared.inplace_predict(Xb)
    p_rank = reg_ranking.predict(xgb.DMatrix(Xb))
    p_bind = clf.inplace_predict(Xb)
    gated = np.where(p_bind >= THRESH, p_sq, p_sq.min() - 1.0)

    brows = [{"model": "reg_squared", **report(yb, p_sq, gb, False)},
             {"model": "reg_ranking", **report(yb, p_rank, gb, True)},
             {"model": "reg_squared + gate", **report(yb, gated, gb, False)}]
    Bdf = pd.DataFrame(brows).set_index("model")
    Bdf.to_csv(OUT / "benchmark_results.csv")
    display(Bdf.round(4))
    print("\nreference, same 500 pairs: Boltz-2 0.412 | PRODIGY-LIG 0.333 | "
          "Savant esm8 0.330 | Savant esm650 0.227 | PLAPT 0.021")

[12:34:51]   DeepChem/ChemBERTa-77M-MLM on cpu, 500 items


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[12:34:54]     256/500


[12:34:55]   Synthyra/ESMplusplus_small on cpu, 500 items


Loading weights:   0%|          | 0/308 [00:00<?, ?it/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


[12:35:07]     8/500


,n,pearson,spearman,between_r,within_r,rmse,rmse_const,pred_std_ratio
model,,,,,,,,
reg_squared,500,0.3426,0.2595,0.3113,0.3454,1.3211,1.273,0.3568
reg_ranking,500,0.1234,0.1053,0.0095,0.3354,NaN,NaN,NaN
reg_squared + gate,500,0.3513,0.2593,0.3754,0.3094,1.3523,1.273,0.4906



reference, same 500 pairs: Boltz-2 0.412 | PRODIGY-LIG 0.333 | Savant esm8 0.330 | Savant esm650 0.227 | PLAPT 0.021


## 10. Save

Schema matches what `ML_model/litpcba/prep_litpcba.py score` already reads
(`classifier_json`, `rdkit_columns`, `ecfp_bits`, `protein_encoder`, `ligand_encoder`), so
the LIT-PCBA screening-deck evaluation runs against this bundle unmodified. Both
regressors are saved; `prep_litpcba.py` only consumes the classifier, since enrichment on
a real deck is a ranking question the classifier's P(binder) already answers.

In [13]:
import joblib

bundle = {
    "variant": "savant_v2_classifier_regressor",
    "protein_encoder": PROTEIN_ENC["hf"], "ligand_encoder": LIGAND_ENC["hf"],
    "block_order": ["ligand_emb", "d11", "rdkit_named", "ecfp", "protein_emb", "aa",
                   "complementarity"],
    "complementarity_names": COMPLEMENT_NAMES,
    "rdkit_columns": RD_COLS, "ecfp_bits": ECFP_BITS,
    "classifier_json": clf.save_raw(raw_format="json"),
    "regressor_squared_json": reg_squared.save_raw(raw_format="json"),
    "regressor_ranking_json": reg_ranking.save_raw(raw_format="json"),
    "binder_pkd": BINDER_PKD, "threshold": THRESH, "target_recall": TARGET_RECALL,
    "caps_as_negatives": USE_CAPS_AS_NEGATIVES,
    "data_sources": {"input_csv": str(SRC), "extra_csv": str(EXTRA) if EXTRA.exists() else None},
    "split": {"mode": "protein+scaffold double-disjoint", "seed": SEED,
              "test_protein_frac": TEST_PROTEIN_FRAC, "test_scaffold_frac": TEST_SCAFFOLD_FRAC,
              "cell_sizes": {k: int(len(v)) for k, v in CELLS.items()}},
    "n_train_pairs": int(len(fit_idx)), "n_train_caps": int(len(cap_fit)),
    "internal_2x2": R.reset_index().to_dict(orient="records"),
    "xgboost_version": xgb.__version__,
}
p = OUT / "savant_v2_bundle.pkl"
joblib.dump(bundle, p, compress=3)
print(f"{p.name}  {p.stat().st_size/1e6:.0f} MB")
print(f"\nScore this against the real screening deck with:")
print(f"    python ../ML_model/litpcba/prep_litpcba.py score --bundle {p} --exclude-functional")

savant_v2_bundle.pkl  61 MB

Score this against the real screening deck with:
    python ../ML_model/litpcba/prep_litpcba.py score --bundle /scratch/sguirale/Savant/ML_model_v2/work/savant_v2_bundle.pkl --exclude-functional
